In [1]:
import pickle

from langdetect import detect

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import ChatOllama

C:\Users\Administrator\AppData\Local\Temp\ipykernel_9376\376943018.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
c:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
with open("documents.pkl", "rb") as f:
    documents = pickle.load(f)

print(f"Documents Loaded: {len(documents)}")

Documents Loaded: 6


In [3]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True}
)

print("Embeddings Loaded!")

Embeddings Loaded!


In [4]:
vectorstore = FAISS.load_local(
    "egypt_faiss",
    embedding_model,
    allow_dangerous_deserialization=True
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k":3}
)

print("FAISS Loaded!")

FAISS Loaded!


In [5]:
llm = ChatOllama(
    model="mistral",
    temperature=0.3
)

print("Mistral Loaded!")

Mistral Loaded!


In [6]:
prompt = ChatPromptTemplate.from_template("""
You are an AI Egypt Tourism Assistant.

Use ONLY the provided context.

Context:
{context}

Question:
{question}

Rules:
- Answer only using the context.
- Do not invent information.
- If the answer is unavailable, reply:
  Not available in the knowledge base.
""")

In [7]:
parser = StrOutputParser()

In [8]:
qa_chain = prompt | llm | parser

In [9]:
def ask_chain(question):

    docs = retriever.invoke(question)

    context = "\n\n".join(
        doc.page_content
        for doc in docs
    )

    response = qa_chain.invoke({
        "context": context,
        "question": question
    })

    return response

In [10]:
print(
    ask_chain(
        "Who built Abu Simbel?"
    )
)

 Pharaoh Ramesses II during the 19th Dynasty, around 1264 to 1244 BCE built Abu Simbel.


In [11]:
print(
    ask_chain(
        "Where is Luxor Temple?"
    )
)

 Luxor Temple is located on the east bank of the Nile, directly in the heart of modern Luxor city.


In [12]:
recommend_prompt = ChatPromptTemplate.from_template("""
You are an Egypt tourism expert.

Use ONLY the context.

Context:
{context}

Interest:
{interest}

Recommend the best tourist attractions.

For each place provide:

- Name
- Why Visit
- Estimated Visit Duration
- Tips
""")

In [13]:
recommend_chain = recommend_prompt | llm | parser

In [14]:
def recommend_places(interest):

    docs = retriever.invoke(interest)

    context = "\n\n".join(
        doc.page_content
        for doc in docs
    )

    return recommend_chain.invoke({
        "context": context,
        "interest": interest
    })

In [15]:
print(
    recommend_places(
        "Ancient Temples"
    )
)

 1. Name: Karnak Temple Complex
   Why Visit: As the largest religious building complex ever constructed, Karnak offers a unique insight into ancient Egyptian architecture and religion. The Precinct of Amun-Re, the Precinct of Mut, the Precinct of Montu, and the Great Hypostyle Hall are must-sees.
   Estimated Visit Duration: 2-3 hours (or more if you wish to explore thoroughly)
   Tips: Wear comfortable shoes as there is a lot of walking involved. Also, bring water and sunscreen, especially during summer months.

2. Name: Luxor Temple
   Why Visit: Located in the heart of modern Luxor city, Luxor Temple offers a chance to witness the continuous use of an ancient structure over 3,400 years, incorporating various religious influences from different eras.
   Estimated Visit Duration: 1-1.5 hours
   Tips: Combine your visit with Luxor Temple with Karnak Temple as they are connected by the Avenue of Sphinxes.

3. Name: Valley of the Kings
   Why Visit: The Valley of the Kings is home to th